In [1]:
%load_ext autoreload
%autoreload 2
 
import itertools
 
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
 
from assignment_1.modules.models import (
    BPRModel,
    PairFeatures,
    TransitionModel,
    apply_rules,
    binary_cosine,
    jaccard,
    transition_rule,
    trend_rule,
    unseen_item_rule,
)
from assignment_1.modules.tasks.task5 import TASK5_BPR_PARAMS
from assignment_1.modules.util import (
    Holdout,
    calc_pair_accuracy,
    load_csv,
    make_pair_validation,
    predict_pairs,
)

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/
 
train = load_csv("task_5_train.csv", DATA_DIR)
test = load_csv("task_5_test.csv", DATA_DIR)

In [3]:
pop = train["item"].value_counts()
p1, p2 = test["item1"].map(pop).fillna(0), test["item2"].map(pop).fillna(0)
print(f"train {train.shape}, test {test.shape} | test users unique: {test.user.is_unique}")
print(f"unseen: users {(~test.user.isin(train.user)).mean():.1%}, "
      f"item1 {(p1 == 0).mean():.1%}, item2 {(p2 == 0).mean():.1%}, "
      f"exactly one {((p1 == 0) ^ (p2 == 0)).mean():.2%}")
print(f"median pop: item1 {p1.median():.0f}, item2 {p2.median():.0f} | "
      f"refs: uniform {pop.median():.0f}, ∝ interactions {train['item'].map(pop).median():.0f}")

train (2554047, 4), test (86786, 4) | test users unique: True
unseen: users 0.0%, item1 2.0%, item2 2.0%, exactly one 3.96%
median pop: item1 101, item2 100 | refs: uniform 2, ∝ interactions 91


In [4]:
tr, pairs = make_pair_validation(train, holdout=Holdout.LAST, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
which = pairs["which"].to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
print(f"{len(pairs):,} validation pairs")

83,508 validation pairs


In [5]:
print(f"tuned BPR: {calc_pair_accuracy(pairs, BPRModel(**TASK5_BPR_PARAMS).fit(tr)):.4f}  (expect 0.8704)")


def bpr_grid(grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        rows.append({**params, "val_acc": calc_pair_accuracy(pairs, BPRModel(**params).fit(tr))})
    return pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)

tuned BPR: 0.8704  (expect 0.8704)


In [6]:
# bpr_grid({"factors": [50, 100], "regularization": [0.005, 0.01, 0.02],
#           "learning_rate": [0.01], "iterations": [100]})

In [7]:
bpr = BPRModel(**TASK5_BPR_PARAMS).fit(tr)
rules_pred = apply_rules(
    predict_pairs(pairs, bpr),
    trend_rule(tr, pairs, min_gap=0.25),
    transition_rule(TransitionModel().fit(tr), pairs, min_gap=5),
    unseen_item_rule(tr, pairs),
)
print(f"rules: all {(rules_pred == which).mean():.4f} | held-out half {(rules_pred[~half] == which[~half]).mean():.4f}")

rules: all 0.8751 | held-out half 0.8728


In [7]:
def evaluate(X: np.ndarray) -> dict:
    out = {}
    for name, m in [("lr", LogisticRegression(max_iter=1000)),
                    ("gb", HistGradientBoostingClassifier(random_state=0))]:
        m.fit(X[half], y[half])
        out[name] = round((m.predict(X[~half]) == y[~half]).mean(), 4)
    return out

In [9]:
X = PairFeatures(TASK5_BPR_PARAMS, transitions=True).fit(tr).transform(pairs)
evaluate(X)        

{'lr': np.float64(0.8913), 'gb': np.float64(0.8932)}

In [10]:
variants = {
    "shipped (no similarity)": None,
    "cosine": (binary_cosine, "cos"),
    "jaccard": (jaccard, "jac"),
}
for name, spec in variants.items():
    pf = (PairFeatures(TASK5_BPR_PARAMS, transitions=True) if spec is None
          else PairFeatures(TASK5_BPR_PARAMS, transitions=True, similarity=spec[0], sim_prefix=spec[1]))
    print(f"{name:<25}", evaluate(pf.fit(tr).transform(pairs)))

shipped (no similarity)   {'lr': np.float64(0.8913), 'gb': np.float64(0.8932)}
cosine                    {'lr': np.float64(0.9001), 'gb': np.float64(0.9029)}
jaccard                   {'lr': np.float64(0.8985), 'gb': np.float64(0.9019)}


In [8]:
from scipy.sparse import csr_matrix

pf = PairFeatures(TASK5_BPR_PARAMS, transitions=True, similarity=binary_cosine, sim_prefix="cos").fit(tr)
X = pf.transform(pairs)                                   # shipped features (with cosine)
sim = pf.sim                                              # ItemSimilarity fitted on tr
ordered = tr.sort_values(["user", "timestamp"], kind="stable")

def recent_sim(m: int) -> np.ndarray:
    """item1 − item2 difference in (max, mean) cosine to each user's last m training books."""
    last = ordered.groupby("user").tail(m)
    Hr = csr_matrix((np.ones(len(last)), (sim.users.get_indexer(last["user"]),
                                          sim.items.get_indexer(last["item"]))), shape=sim.R.shape)
    Hr.data[:] = 1
    u = sim.users.get_indexer(pairs["user"])

    def feats(items):
        i = sim.items.get_indexer(items)
        ok = (u >= 0) & (i >= 0)
        out = np.zeros((len(u), 2))
        P = Hr[u[ok]].multiply(sim.S[i[ok]]).tocsr()
        out[ok, 0] = P.max(axis=1).toarray().ravel()
        out[ok, 1] = np.asarray(P.sum(axis=1)).ravel() / m
        return out

    return feats(pairs["item1"]) - feats(pairs["item2"])

for m in [1, 3, 5, 10]:
    print(f"+ recent cosine, m={m:<3}", evaluate(np.column_stack([X, recent_sim(m)])))

+ recent cosine, m=1   {'lr': np.float64(0.9005), 'gb': np.float64(0.9025)}
+ recent cosine, m=3   {'lr': np.float64(0.9004), 'gb': np.float64(0.9026)}
+ recent cosine, m=5   {'lr': np.float64(0.9007), 'gb': np.float64(0.9031)}
+ recent cosine, m=10  {'lr': np.float64(0.9005), 'gb': np.float64(0.9023)}


In [9]:
small = BPRModel(factors=10, regularization=0.01, learning_rate=0.01, iterations=100).fit(tr)
uu = pairs["user"].to_numpy()
d_small = small(uu, pairs["item1"].to_numpy()) - small(uu, pairs["item2"].to_numpy())
print("+ BPR k=10            ", evaluate(np.column_stack([X, d_small])))

+ BPR k=10             {'lr': np.float64(0.8998), 'gb': np.float64(0.9028)}


In [ ]:
print("+ recent (best m) + BPR k=10", evaluate(np.column_stack([X, recent_sim(BEST_M), d_small])))